# Feature reranker over beams, cross-fitted on dev (CPU)

**Question.** A correct program is among the 4 beams for 713 of 883 dev questions, but the top beam
is correct for 625, and fixed rules recovered none (Notebook 22). Does a simple learned reranker over
label-free beam features pick better?

**Protocol (pre-registered in the experiment log before any fit).** Features per beam: rank,
log-prob, gap to the top beam, executes, grounded, agreement with the other beams and with the 16
samples, steps, table use, constant use, literals (`finevidence.generation.reranker`). Model:
standardized logistic regression, C = 1.0, no tuning. 5-fold cross-fitting over questions (seed 42):
each question is predicted by a model fitted on the other 4 folds. Features are computed and hashed
before labels are read. Exact two-sided McNemar against the top beam (625/883). Test is not loaded.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import subprocess
import time
from pathlib import Path

import numpy as np
from scipy.stats import binomtest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from finevidence.data import load_finqa_split
from finevidence.generation import load_rankings
from finevidence.generation.inputs import build_keyed_example
from finevidence.generation.reranker import FEATURE_NAMES, beam_features

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
processed = PROJECT_ROOT / "data" / "processed"
EXPECTED = {
    "generator-qwen2.5-1.5b-lora-top10-v1_dev_candidates.jsonl": (
        "237557f60207ee8876a506525791a0db3926f9f4ad736125ae4f89ef7fd38c92"
    ),
    "generator-qwen2.5-1.5b-lora-top10-v1_dev_retrieved_top_beam_predictions.jsonl": (
        "6a8b8f9d8f165bc14b19d360ac6720f05aa877f9dbd4c31b2b705426f84ec418"
    ),
    "retriever_bert_base_dev_rankings.jsonl": (
        "d6c31f86c1fd4ab0d68d2cb56996ffe9705fba7dfbfe62464fd7b0ea7bd562e6"
    ),
}
for name, sha in EXPECTED.items():
    assert hashlib.sha256((processed / name).read_bytes()).hexdigest() == sha, name
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
print({"commit": commit, "python": platform.python_version(), "inputs_verified": True})

{'commit': '45d072020fffc0e00ae7a8d12c0c820034c5e886', 'python': '3.13.16', 'inputs_verified': True}


## 1. Audit: alignment and provenance (as in Notebook 22)

In [2]:
dev = load_finqa_split(PROJECT_ROOT / "data" / "raw" / "dev.json")
rankings = load_rankings(processed / "retriever_bert_base_dev_rankings.jsonl")
candidates = [
    json.loads(line)
    for line in (processed / "generator-qwen2.5-1.5b-lora-top10-v1_dev_candidates.jsonl")
    .read_text()
    .splitlines()
]
baseline = [
    json.loads(line)
    for line in (
        processed / "generator-qwen2.5-1.5b-lora-top10-v1_dev_retrieved_top_beam_predictions.jsonl"
    )
    .read_text()
    .splitlines()
]
assert [row["record_id"] for row in candidates] == [str(record["id"]) for record in dev]
assert [row["record_id"] for row in baseline] == [row["record_id"] for row in candidates]
assert {row["commit"] for row in candidates} == {row["commit"] for row in baseline}
assert all(row["fact_keys"] == rankings[row["record_id"]][:10] for row in candidates)
assert all(
    c["beam_candidates"][0] == b["prediction"] for c, b in zip(candidates, baseline, strict=True)
)
assert all(
    len(row["beam_candidates"]) == len(row["beam_scores"]) == 4
    and len(row["sample_candidates"]) == len(row["sample_scores"]) == 16
    for row in candidates
)
print(
    {
        "records": len(candidates),
        "generator_commit": candidates[0]["commit"],
        "fact_keys_are_ranking_top10": True,
        "beam0_is_baseline": True,
    }
)

{'records': 883, 'generator_commit': '6cc650e002f1851f1801346ad9d17289fb080282', 'fact_keys_are_ranking_top10': True, 'beam0_is_baseline': True}


## 2. Label-free features

In [3]:
label_free = [
    {
        "id": r["id"],
        "table": r["table"],
        "pre_text": r["pre_text"],
        "post_text": r["post_text"],
        "qa": {"question": r["qa"]["question"]},
    }
    for r in dev
]
fact_texts = [
    build_keyed_example(record, row["fact_keys"], None).fact_texts
    for record, row in zip(label_free, candidates, strict=True)
]
started = time.perf_counter()
features = np.array(
    [
        beam_features(
            row["beam_candidates"],
            row["beam_scores"],
            row["sample_candidates"],
            facts,
            record["table"],
        )
        for row, facts, record in zip(candidates, fact_texts, label_free, strict=True)
    ]
)  # questions x 4 beams x features
feature_ms = (time.perf_counter() - started) / len(candidates) * 1000
features_sha = hashlib.sha256(features.tobytes()).hexdigest()
print(
    {
        "shape": features.shape,
        "features_sha256": features_sha,
        "cpu_ms_per_question": round(feature_ms, 2),
    }
)

{'shape': (883, 4, 11), 'features_sha256': '2355a4ebf44e6b7cac738469be5d87d79a25a8c0775686550b839097eede37d1', 'cpu_ms_per_question': 0.59}


## 3. Labels (read only now) and out-of-fold selection

In [4]:
from finevidence.generation import score_program_prediction  # noqa: E402

assert hashlib.sha256(features.tobytes()).hexdigest() == features_sha
labels = np.array(
    [
        [
            score_program_prediction(record, p, facts).execution_correct
            for p in row["beam_candidates"]
        ]
        for record, row, facts in zip(dev, candidates, fact_texts, strict=True)
    ]
)
chosen = np.zeros(len(dev), dtype=int)
folds = KFold(n_splits=5, shuffle=True, random_state=42)
for fit_idx, held_idx in folds.split(features):
    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=1000))
    model.fit(features[fit_idx].reshape(-1, features.shape[2]), labels[fit_idx].reshape(-1))
    probs = model.predict_proba(features[held_idx].reshape(-1, features.shape[2]))[:, 1]
    chosen[held_idx] = probs.reshape(len(held_idx), -1).argmax(axis=1)  # ties -> lower rank
coefficients = dict(zip(FEATURE_NAMES, np.round(model[-1].coef_[0], 3).tolist(), strict=True))
print({"last_fold_coefficients": coefficients})

{'last_fold_coefficients': {'rank': -0.104, 'log_prob': 0.813, 'gap_to_top': -0.211, 'executes': 0.268, 'grounded': 0.025, 'beam_agreement': 0.601, 'sample_agreement': 1.499, 'steps': -0.011, 'uses_table': -0.005, 'uses_constant': -0.279, 'literals': -0.19}}


## 4. Comparison with the top beam

In [5]:
old = np.array([row["execution_correct"] for row in baseline])
assert old.sum() == 625 and (labels[:, 0] == old).all()
new = labels[np.arange(len(dev)), chosen]
fixed, broken = int((new & ~old).sum()), int((old & ~new).sum())
program_new = [
    score_program_prediction(record, row["beam_candidates"][i], facts).program_correct
    for record, row, facts, i in zip(dev, candidates, fact_texts, chosen, strict=True)
]
result = {
    "execution_correct": int(new.sum()),
    "execution_accuracy": round(float(new.mean()), 4),
    "program_accuracy": round(float(np.mean(program_new)), 4),
    "changed": int((chosen != 0).sum()),
    "fixed": fixed,
    "broken": broken,
    "net": fixed - broken,
    "mcnemar_exact_p": round(float(binomtest(fixed, fixed + broken).pvalue), 6)
    if fixed + broken
    else 1.0,
    "beam_oracle": int(labels.any(axis=1).sum()),
    "cpu_ms_per_question": round(feature_ms, 2),
}
print(json.dumps(result, indent=2))
(processed / "reranker-beams4_dev_report.json").write_text(
    json.dumps(
        {
            "commit": commit,
            "features_sha256": features_sha,
            "result": result,
            "chosen_beam": chosen.tolist(),
        },
        indent=2,
    )
)

{
  "execution_correct": 625,
  "execution_accuracy": 0.7078,
  "program_accuracy": 0.658,
  "changed": 62,
  "fixed": 13,
  "broken": 13,
  "net": 0,
  "mcnemar_exact_p": 1.0,
  "beam_oracle": 713,
  "cpu_ms_per_question": 0.59
}


6617